# Regressão logística

**Objetivo:** treinar um classificador binário, ler os coeficientes como razões de chance, desenhar a fronteira de decisão e ver que 'logística' e 'uma camada linear + sigmoide' são a mesma coisa — o mesmo modelo, no scikit-learn e num laço explícito de PyTorch.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## 1. Dados: diagnóstico de câncer de mama

Conjunto **breast cancer** do scikit-learn: 30 medidas de imagens de núcleos celulares, alvo binário (maligno = 1, benigno = 0).

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

dados = load_breast_cancer(as_frame=True)
X = dados.data.values
y = dados.target.values
# nesta base, 0 = maligno e 1 = benigno; invertemos para 1 = maligno (o positivo de interesse)
y = 1 - y
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=SEMENTE, stratify=y)
escala = StandardScaler().fit(X_tr)
X_tr = escala.transform(X_tr)
X_te = escala.transform(X_te)
print("treino:", X_tr.shape, "| malignos no treino:", int(y_tr.sum()))

## 2. Ajuste com o scikit-learn e razões de chance

O coeficiente $\theta_j$ vira a razão de chances $e^{\theta_j}$: quanto as chances de ser maligno mudam ao aumentar aquele preditor em um desvio-padrão (os dados estão padronizados).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

modelo = LogisticRegression(max_iter=5000)
modelo.fit(X_tr, y_tr)
proba = modelo.predict_proba(X_te)[:, 1]
previsto = (proba > 0.5).astype(int)
print("acuracia:", round(accuracy_score(y_te, previsto), 3))
print("AUC:", round(roc_auc_score(y_te, proba), 3))

razao_chances = np.exp(modelo.coef_[0])
ordem = np.argsort(razao_chances)[::-1]
print("\ntres maiores fatores de risco (razao de chances):")
for j in ordem[:3]:
    print("  ", dados.data.columns[j], "->", round(razao_chances[j], 2))

## 3. A fronteira de decisão em 2D

Para visualizar, treinamos de novo com só dois preditores e pintamos a região que o modelo chama de maligna.

In [ ]:
col_a, col_b = 20, 27   # dois preditores (raio e concavidade "piores")
X2 = X_tr[:, [col_a, col_b]]
modelo2 = LogisticRegression(max_iter=5000).fit(X2, y_tr)

passo = 0.05
gx, gy = np.meshgrid(np.arange(X2[:, 0].min()-1, X2[:, 0].max()+1, passo),
                     np.arange(X2[:, 1].min()-1, X2[:, 1].max()+1, passo))
grade = np.c_[gx.ravel(), gy.ravel()]
zz = modelo2.predict_proba(grade)[:, 1].reshape(gx.shape)

figura = go.Figure()
figura.add_trace(go.Contour(x=gx[0], y=gy[:, 0], z=zz, showscale=False,
                            colorscale=[[0, "#dce7f4"], [1, "#f6dedb"]], opacity=0.7,
                            contours=dict(start=0.5, end=0.5, size=1, coloring="lines")))
figura.add_trace(go.Scatter(x=X2[y_tr==0, 0], y=X2[y_tr==0, 1], mode="markers",
                            marker=dict(color=AZUL, size=6), name="benigno"))
figura.add_trace(go.Scatter(x=X2[y_tr==1, 0], y=X2[y_tr==1, 1], mode="markers",
                            marker=dict(color=VERMELHO, size=6), name="maligno"))
figura.update_layout(title="Fronteira de decisao (dois preditores)",
                     height=420, margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 4. A mesma logística, à mão em PyTorch

Uma camada linear seguida de sigmoide **é** uma regressão logística. Treinamos com o laço explícito de sempre (forward → custo → backward → passo) e vemos a acurácia bater com a do scikit-learn.

In [ ]:
import torch

entradas = torch.tensor(X_tr, dtype=torch.float32)
alvos = torch.tensor(y_tr, dtype=torch.float32).reshape(-1, 1)

torch.manual_seed(SEMENTE)
rede = torch.nn.Sequential(torch.nn.Linear(X_tr.shape[1], 1), torch.nn.Sigmoid())
custo_fn = torch.nn.BCELoss()
otimizador = torch.optim.Adam(rede.parameters(), lr=0.05)

historico = []
for epoca in range(300):
    previsto_t = rede(entradas)          # forward
    custo = custo_fn(previsto_t, alvos)   # custo
    otimizador.zero_grad()
    custo.backward()                      # backward
    otimizador.step()                     # passo
    historico.append(custo.item())

with torch.no_grad():
    proba_te = rede(torch.tensor(X_te, dtype=torch.float32)).numpy().ravel()
acc_torch = ((proba_te > 0.5).astype(int) == y_te).mean()
print("acuracia da rede (PyTorch):", round(float(acc_torch), 3))
print("acuracia do sklearn:      ", round(accuracy_score(y_te, previsto), 3))

In [ ]:
figura = go.Figure(go.Scatter(y=historico, mode="lines", line=dict(color=VERDE)))
figura.update_layout(title="Custo (log-loss) do treino em PyTorch",
                     xaxis_title="epoca", yaxis_title="BCE", height=320,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## Exercício

A matriz de confusão abaixo separa os erros em falsos positivos e falsos negativos. Em rastreio de câncer, qual dos dois é mais grave? Como o limiar de 0,5 poderia ser ajustado para reduzi-lo?

In [ ]:
# @title Solução (clique para revelar)
mc = confusion_matrix(y_te, previsto)
print("matriz de confusao [linhas=verdade, colunas=previsto]:")
print(mc)
print("\nfalsos negativos (maligno dito benigno):", mc[1, 0])
# Em rastreio, o FALSO NEGATIVO (deixar passar um maligno) costuma ser o mais
# grave. Baixar o limiar (ex.: prever maligno se proba > 0.3) captura mais
# malignos, ao custo de mais falsos positivos (mais biopsias de confirmacao).